In [2]:

import librosa
import numpy as np
import noisereduce as nr

# Methods to process the audio to model input format

FIXED_LENGTH = 128
N_BINS = 84
MIN_SEGMENT_LENGTH = 1024

# Process real audio from wav file
def create_cqt(y, sr, hop_length=512):
    cqt = librosa.cqt(y, sr=sr, hop_length=hop_length, n_bins=N_BINS, bins_per_octave=12)
    cqt_db = librosa.amplitude_to_db(np.abs(cqt), ref=np.max)
    cqt_db = np.maximum(cqt_db, -80)  
    return cqt_db

def pad_or_truncate(cqt, fixed_length=FIXED_LENGTH):
    if cqt.shape[1] >= fixed_length:
        return cqt[:, :fixed_length]
    pad_width = fixed_length - cqt.shape[1]
    return np.pad(cqt, ((0, 0), (0, pad_width)))



In [3]:
# Try number #1 with raw audio (no noise reduction)
def process_recording(audio_path, delta=0.07, wait=8, pre_max=3, post_max=3):
    y, sr = librosa.load(audio_path, sr=22050)

    onset_frames = librosa.onset.onset_detect(
        y=y, sr=sr,
        delta=delta,       
        wait=wait,         
        pre_max=pre_max,
        post_max=post_max,
        backtrack=True    
    )
    onset_samples = librosa.frames_to_samples(onset_frames)
    onset_times = librosa.frames_to_time(onset_frames, sr=sr)

    features = []
    valid_onset_times = []

    # For each note sampled, get the spectogram/features
    for i, start in enumerate(onset_samples):
        end = onset_samples[i + 1] if i + 1 < len(onset_samples) else len(y)
        segment = y[start:end]

        if len(segment) < MIN_SEGMENT_LENGTH:
            continue  

        cqt = create_cqt(segment, sr)
        cqt_fixed = pad_or_truncate(cqt)
        features.append(cqt_fixed)
        valid_onset_times.append(onset_times[i])

    features = np.stack(features) if features else np.empty((0, N_BINS, FIXED_LENGTH))
    return features, valid_onset_times

In [6]:
features, onset_times = process_recording("./process_recording_files/happy_birthday.wav")
print(f"Extracted {len(features)} note segments")
print(f"Features shape: {features.shape}")  # (61, 84, 128)

# Extracts 7 notes, there should be 4



Extracted 7 note segments
Features shape: (7, 84, 128)


c:\Users\megan\AppData\Local\Programs\Python\Python311\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=192
  warnings.warn(
c:\Users\megan\AppData\Local\Programs\Python\Python311\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=96
  warnings.warn(
c:\Users\megan\AppData\Local\Programs\Python\Python311\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=128
  warnings.warn(
c:\Users\megan\AppData\Local\Programs\Python\Python311\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=248
  warnings.warn(
c:\Users\megan\AppData\Local\Programs\Python\Python311\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=240
  warnings.warn(
c:\Users\megan\AppData\Local\Programs\Python\Python311\Lib\site-pa

In [27]:
features, onset_times = process_recording("./process_recording_files/fingerpicking.wav")
print(f"Extracted {len(features)} note segments")
print(f"Features shape: {features.shape}")  # (61, 84, 128)
# Extracts more notes

Extracted 17 note segments
Features shape: (17, 84, 128)


# Predict the notes

In [11]:
from model import CNN
import torch
test_model = CNN(num_notes=61)
test_model.load_state_dict(torch.load('note_classifier.pth'))

<All keys matched successfully>

In [12]:
from music21 import stream, note, tempo, metadata
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# Be able to read all the notes

df = pd.read_csv('./test_files/labels.csv')
le_note = LabelEncoder()
y_note = le_note.fit_transform(df['note'].values)

classes = le_note.classes_

print(classes)

c:\Users\megan\AppData\Local\Programs\Python\Python311\Lib\site-packages\requests\__init__.py:109: RequestsDependencyWarning: urllib3 (2.0.4) or chardet (7.4.0.post2)/charset_normalizer (3.2.0) doesn't match a supported version!
  warnings.warn(


['A#2' 'A#3' 'A#4' 'A#5' 'A#6' 'A2' 'A3' 'A4' 'A5' 'A6' 'B2' 'B3' 'B4'
 'B5' 'B6' 'C#2' 'C#3' 'C#4' 'C#5' 'C#6' 'C2' 'C3' 'C4' 'C5' 'C6' 'C7'
 'D#2' 'D#3' 'D#4' 'D#5' 'D#6' 'D2' 'D3' 'D4' 'D5' 'D6' 'E2' 'E3' 'E4'
 'E5' 'E6' 'F#2' 'F#3' 'F#4' 'F#5' 'F#6' 'F2' 'F3' 'F4' 'F5' 'F6' 'G#2'
 'G#3' 'G#4' 'G#5' 'G#6' 'G2' 'G3' 'G4' 'G5' 'G6']


In [29]:
import numpy as np
def predict_notes(features: np.ndarray):
    # Ensure it's a plain numpy array of the right dtype first
    features = np.array(features, dtype=np.float32)  # force correct type
    
    # Now convert to tensor and add channel dim
    tensor = torch.from_numpy(features).unsqueeze(1)  # (N, 1, 84, 128)
    
    print(type(tensor), tensor.dtype, tensor.shape)  # verify before model call
    
    with torch.no_grad():
        outputs = test_model(tensor)
        # print(outputs)
        # indices = outputs.argmax(dim=1) # just getting the max value
        probs = torch.softmax(outputs, dim=1)
        # print(probs)
        max_probs, indices = probs.max(dim=1)

        # only keep predictions the model is confident about
        CONFIDENCE_THRESHOLD = 0.4
        predicted_notes = [
            idx if prob.item() > CONFIDENCE_THRESHOLD else None
            for idx, prob in zip(indices, max_probs)
        ]

    return predicted_notes

In [30]:
import os

# Generate wave file (modified from evaluation.py converting to midi)
def generate_wav(preds, le_note, rhythm_preds=None, le_duration=None,
                 filename="predicted_output.wav",
                 soundfont_path="GeneralUser-GS/GeneralUser-GS.sf2"):
    s = stream.Stream()
    s.append(tempo.MetronomeMark(number=120))

    classes = le_note.classes_
    print(classes)

    for i, idx in enumerate(preds):
        try:
            n = note.Note(str(classes[idx]))
            if rhythm_preds is not None and le_duration is not None:
                n.duration.quarterLength = float(le_duration.classes_[rhythm_preds[i]])
            else:
                n.duration.quarterLength = 1.0
            s.append(n)
        except Exception:
            continue

    # Write to midi first, then convert to wav via FluidSynth
    midi_path = filename.replace('.wav', '.mid')
    s.write('midi', fp=midi_path)
    os.system(f'fluidsynth -ni -F "{filename}" "{soundfont_path}" "{midi_path}"')
    os.remove(midi_path)  # clean up intermediate midi

In [31]:
# predict the features
notes = predict_notes(features)
print(notes)

<class 'torch.Tensor'> torch.float32 torch.Size([17, 1, 84, 128])
[tensor(41), tensor(56), tensor(56), tensor(22), tensor(56), tensor(48), tensor(11), tensor(60), tensor(41), tensor(56), tensor(33), tensor(41), tensor(33), tensor(22), tensor(46), None, None]


In [32]:
output_file = "test_prediction_fingerpicking.wav"

generate_wav([note for note in notes if note], le_note, filename=output_file)


['A#2' 'A#3' 'A#4' 'A#5' 'A#6' 'A2' 'A3' 'A4' 'A5' 'A6' 'B2' 'B3' 'B4'
 'B5' 'B6' 'C#2' 'C#3' 'C#4' 'C#5' 'C#6' 'C2' 'C3' 'C4' 'C5' 'C6' 'C7'
 'D#2' 'D#3' 'D#4' 'D#5' 'D#6' 'D2' 'D3' 'D4' 'D5' 'D6' 'E2' 'E3' 'E4'
 'E5' 'E6' 'F#2' 'F#3' 'F#4' 'F#5' 'F#6' 'F2' 'F3' 'F4' 'F5' 'F6' 'G#2'
 'G#3' 'G#4' 'G#5' 'G#6' 'G2' 'G3' 'G4' 'G5' 'G6']
